In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0"
}

url = "https://www.thequint.com/news/webqoof"

r = requests.get(url, headers=headers)

print(r.status_code)

200


In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

links = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    if "/news/webqoof/" in href:
        links.append(href)

links = list(dict.fromkeys(links))

print(len(links))
print(links[:10])

10
['https://www.thequint.com/news/webqoof/unrelated-video-falsely-shared-as-one-of-nihang-sikhs-brandishing-swords-against-police-cjp-protest-fact-check', 'https://www.thequint.com/news/webqoof/delhi-police-pro-say-that-amit-shah-ordered-them-to-end-cjp-protests-false-claim-fact-check', 'https://www.thequint.com/news/webqoof/muslim-man-trying-to-set-car-on-fire-video-is-from-bihar-not-cjp-protest-delhi-fact-check', 'https://www.thequint.com/news/webqoof/the-video-show-police-disrespecting-indian-flag-during-cjp-protests-false-claim-fact-check', 'https://www.thequint.com/news/webqoof/unrelated-video-falsely-shared-as-police-vandalising-a-car-during-cjp-protests-false-claim-fact-check', 'https://www.thequint.com/news/webqoof/unrelated-video-shared-as-cjp-protestors-pelting-stones-at-a-new-hydrogen-train-false-claim-fact-check', 'https://www.thequint.com/news/webqoof/bangladesh-jamaat-e-islami-cjp-support-letter-ai-generated-fake-fact-check', 'https://www.thequint.com/news/webqoof/old-un

In [ ]:
url = links[0]

r = requests.get(url, headers=headers)

print(r.status_code)

soup = BeautifulSoup(r.text, "html.parser")

200


In [ ]:
title = soup.find("h1")

print(title.get_text(strip=True))

Unrelated Video Shared as One of Nihang Sikhs Using Swords During CJP Protest


In [ ]:
meta = soup.find(
    "meta",
    attrs={"property":"article:published_time"}
)

print(meta["content"] if meta else None)

2026-07-23T18:40:35+05:30


In [ ]:
paragraphs = soup.find_all("p")

content = "\n".join(
    p.get_text(" ", strip=True)
    for p in paragraphs
)

print(content[:1000])

A video showing Nihang Sikhs , who belong to a martial order of Sikhism and are characterised by their deep blue robes and turban, pushing against barricades is being widely shared on social media.
The claim: Those sharing this video have claimed that it shows visuals of Nihang Sikhs brandishing swords against the Delhi Police during the Cockroach Janta Party -led (CJP) protest, which took a violent turn.
(Archived versions of more posts sharing this video as one from the recent protests can be seen here , here , and here .)
But...?: The claim is misleading, because the video is from Uttarakhand's border and has no connection to the protests in Delhi.
How did we find out the truth?: We noticed that one barricade that was being pushed read 'Dehradun Traffic Police', indicating that the video may be from Uttarakhand.
We then ran a reverse image search on the video, which led us to an Instagram post shared on 26 June 2026 by BBC News Punjabi, which showed the same visuals.
Their post ment

In [ ]:
import requests
from bs4 import BeautifulSoup

xml = requests.get("https://www.thequint.com/sitemap.xml").text

soup = BeautifulSoup(xml, "xml")

daily = [
    x.text for x in soup.find_all("loc")
    if "sitemap-daily" in x.text
]

print(len(daily))
print(daily[:5])

4177
['https://www.thequint.com/sitemap/sitemap-daily-2026-07-23.xml', 'https://www.thequint.com/sitemap/sitemap-daily-2026-07-22.xml', 'https://www.thequint.com/sitemap/sitemap-daily-2026-07-21.xml', 'https://www.thequint.com/sitemap/sitemap-daily-2026-07-20.xml', 'https://www.thequint.com/sitemap/sitemap-daily-2026-07-19.xml']


In [ ]:
# Keep only 2022–2025 daily sitemaps
daily = [
    x for x in daily
    if any(f"sitemap-daily-{year}-" in x for year in [2022, 2023, 2024, 2025])
]

print(len(daily))

1461


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

posts = []

for sm in tqdm(daily):

    try:
        xml = requests.get(sm, timeout=20).text
        soup = BeautifulSoup(xml, "xml")

        for u in soup.find_all("url"):

            posts.append({
                "url": u.loc.text,
                "lastmod": u.lastmod.text
            })

    except Exception:
        continue

posts = pd.DataFrame(posts)

print(posts.shape)

100%|██████████| 1461/1461 [03:46<00:00,  6.46it/s]


(69906, 2)


In [ ]:
posts = posts[
    posts["url"].str.contains("/news/webqoof/", na=False)
]

posts = posts.drop_duplicates(subset="url")

print(posts.shape)
posts.head()

(4737, 2)


,url,lastmod
7,https://www.thequint.com/news/webqoof/unrelate...,2025-12-31T12:49:25.508+05:30
19,https://www.thequint.com/news/webqoof/pm-naren...,2025-12-30T13:26:58.611+05:30
21,https://www.thequint.com/news/webqoof/aditya-r...,2025-12-30T12:32:25.557+05:30
31,https://www.thequint.com/news/webqoof/no-kulde...,2025-12-29T16:32:16.916+05:30
34,https://www.thequint.com/news/webqoof/ai-gener...,2025-12-29T14:33:29.079+05:30


In [ ]:
posts = posts.drop_duplicates(subset="url")

print(posts.shape)

(4737, 2)


In [ ]:
posts["lastmod"] = pd.to_datetime(posts["lastmod"])

posts["year"] = posts["lastmod"].dt.year

print(posts["year"].value_counts().sort_index())

year
2022     981
2023    1098
2024    1347
2025    1311
Name: count, dtype: int64


In [ ]:
posts.sample(20, random_state=42)["url"].tolist()

['https://www.thequint.com/news/webqoof/webqoof-recap-30-march-to-5-april-arvind-kejriwal-media-misreporting-vaiko-rahul-gandhi-ipl',
 'https://www.thequint.com/news/webqoof/russian-president-vladimir-putin-indian-national-anthem-edited-video-fact-check',
 'https://www.thequint.com/news/webqoof/cow-shot-by-man-manipur-old-video-not-wayanad-kerala-muslim-fact-check',
 'https://www.thequint.com/news/webqoof/unrelated-photo-shared-as-recent-photo-of-protests-in-brazil-fact-check',
 'https://www.thequint.com/news/webqoof/2019-survey-figures-misrepresented-as-rahul-gandhi-popularity-bharat-jodo-yatra-fact-check',
 'https://www.thequint.com/news/webqoof/aimim-asaduddin-owaisi-refused-i-love-muhammad-photo-frame-viral-video-fact-check',
 'https://www.thequint.com/news/webqoof/virat-kohli-anushka-sharma-family-photo-is-ai-generated-fact-check',
 'https://www.thequint.com/news/webqoof/ai-generated-images-of-shah-rukh-khan-ronda-rousey-roman-reigns-at-kumbh-mela-viral-fact-check',
 'https://www.

In [ ]:
url = posts.sample(1, random_state=1)["url"].iloc[0]

r = requests.get(url, headers=headers)

print(r.status_code)

soup = BeautifulSoup(r.text, "html.parser")

title = soup.find("h1")
print(title.get_text(strip=True))

paragraphs = soup.find_all("p")

content = "\n".join(
    p.get_text(" ", strip=True)
    for p in paragraphs
)

print(len(content))
print(content[:500])

200
No, Raghuram Rajan Is Not the New Governor of the Bank of England
2658
A photo of former Reserve Bank of India ( RBI ) Governor Raghuram Rajan is being shared on social media, accompanied by a text which congratulates him on being appointed as the new governor of the Bank of England.
However, we found that the claim is false. The bank's website shows Andrew Bailey as its governor who took charge in 2020. Bailey was also referred to as the governor in a recent press conference held on 5 May 2022.
Moreover, this claim has been doing the rounds since 2018.
CLAIM
The p


In [ ]:
import os
import pandas as pd

checkpoint_file = "webqoof_checkpoint.csv"

if os.path.exists(checkpoint_file):

    checkpoint = pd.read_csv(checkpoint_file)

    print(f"Found checkpoint with {len(checkpoint)} articles.")

else:

    checkpoint = pd.DataFrame()

    print("No checkpoint found.")

No checkpoint found.


In [ ]:
import os

checkpoint_file = "webqoof_checkpoint.csv"

if os.path.exists(checkpoint_file):

    webqoof = pd.read_csv(checkpoint_file)

    articles = webqoof.to_dict("records")

    scraped_urls = set(webqoof["url"])

    posts = posts[~posts["url"].isin(scraped_urls)]

    print(f"Resuming from {len(articles)} articles.")

else:

    articles = []

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import time

#----------------------------------------------------------
# Browser headers
#----------------------------------------------------------

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

articles = []

#----------------------------------------------------------
# Scrape each WebQoof article
#----------------------------------------------------------

for i, row in enumerate(tqdm(posts.itertuples(index=False),
                             total=len(posts)), start=1):

    url = row.url

    try:

        r = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        #--------------------------------------------------
        # Title
        #--------------------------------------------------

        title = soup.find("h1")
        title = title.get_text(strip=True) if title else None

        #--------------------------------------------------
        # Published date
        #--------------------------------------------------

        meta = soup.find(
            "meta",
            attrs={"property": "article:published_time"}
        )

        if meta:
            date = meta["content"][:10]
        else:
            date = row.lastmod[:10]

        #--------------------------------------------------
        # Article content
        #--------------------------------------------------

        paragraphs = soup.find_all("p")

        content = "\n".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
        )

        #--------------------------------------------------
        # Save article
        #--------------------------------------------------

        articles.append({

            "title": title,
            "url": url,
            "date": date,
            "content": content

        })

        #--------------------------------------------------
        # Checkpoint every 50 articles
        #--------------------------------------------------

        if i % 50 == 0:

            pd.DataFrame(articles).to_csv(
                "webqoof_checkpoint.csv",
                index=False
            )

        time.sleep(0.5)

    except Exception as e:

        print(f"Error on article {i}")
        print(url)
        print(e)

        continue

#----------------------------------------------------------
# Final dataset
#----------------------------------------------------------

webqoof = pd.DataFrame(articles)

webqoof.to_csv(
    "webqoof_2022_2025.csv",
    index=False
)

print("Finished!")

print("Articles scraped:", len(webqoof))

 36%|███▌      | 1713/4737 [20:15<30:18,  1.66it/s]

Error on article 1713
https://www.thequint.com/news/webqoof/russian-accounts-targeting-fact-checkers-by-bombarding-fake-content-fabricated-news-reports-2
'Timestamp' object is not subscriptable


 49%|████▉     | 2329/4737 [27:37<22:43,  1.77it/s]

Error on article 2329
https://www.thequint.com/news/webqoof/hall-of-fake-news-fact-check-fake-claims
'Timestamp' object is not subscriptable


 61%|██████    | 2900/4737 [34:28<18:10,  1.68it/s]

...

Error on article 3695
https://www.thequint.com/news/webqoof/boycott-pathaan-protest-shah-rukh-khan-deepika-padukone-part-1
'Timestamp' object is not subscriptable


 78%|███████▊  | 3708/4737 [44:18<11:34,  1.48it/s]

Error on article 3708
https://www.thequint.com/news/webqoof/the-making-of-boycott-pathaan-uncovering-the-reality-behind-outrage-against-shah-rukh-khan
'Timestamp' object is not subscriptable


100%|██████████| 4737/4737 [56:46<00:00,  1.39it/s]


Finished!
Articles scraped: 4725


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# -------------------------------
# Failed URLs
# -------------------------------

failed_urls = [
    "https://www.thequint.com/news/webqoof/russian-accounts-targeting-fact-checkers-by-bombarding-fake-content-fabricated-news-reports-2",
    "https://www.thequint.com/news/webqoof/hall-of-fake-news-fact-check-fake-claims",
    "https://www.thequint.com/news/webqoof/online-shopping-scams-e-commerce-frauds-scamguard-guide",
    "https://www.thequint.com/news/webqoof/online-scammers-duping-people-of-money-using-upi-frauds-scamguard-guide",
    "https://www.thequint.com/news/webqoof/how-to-check-tackle-online-job-scams-frauds-guide-social-media",
    "https://www.thequint.com/news/webqoof/scamguard-interactive-immersive-guides-how-to-tackle-online-scams-digital-frauds",
    "https://www.thequint.com/news/webqoof/how-to-detect-artificial-intelligence-ai-images-real-photos-or-fake-cheatsheet",
    "https://www.thequint.com/news/webqoof/how-chatgpt-artificial-intelligence-can-spread-misinformation-on-the-internet-openai-gpt4",
    "https://www.thequint.com/news/webqoof/boycott-pathaan-trend-outrage-shah-rukh-khan-deepika-padukone-key-takeaways-highlights",
    "https://www.thequint.com/news/webqoof/behind-boycott-pathaan-trend-outrage-against-shah-rukh-khan-deepika-padukone-social-media-players",
    "https://www.thequint.com/news/webqoof/boycott-pathaan-protest-shah-rukh-khan-deepika-padukone-part-1",
    "https://www.thequint.com/news/webqoof/the-making-of-boycott-pathaan-uncovering-the-reality-behind-outrage-against-shah-rukh-khan"
]

# Keep only those rows
failed_posts = posts[posts["url"].isin(failed_urls)]

missing_articles = []

for row in tqdm(failed_posts.itertuples(index=False), total=len(failed_posts)):

    try:

        r = requests.get(row.url, headers=headers, timeout=20)

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        # Title
        title = soup.find("h1")
        title = title.get_text(strip=True) if title else None

        # Date
        meta = soup.find(
            "meta",
            attrs={"property": "article:published_time"}
        )

        if meta:
            date = meta["content"][:10]
        else:
            # FIXED
            date = row.lastmod.strftime("%Y-%m-%d")

        # Content
        paragraphs = soup.find_all("p")

        content = "\n".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
        )

        missing_articles.append({
            "title": title,
            "url": row.url,
            "date": date,
            "content": content
        })

        time.sleep(0.5)

    except Exception as e:

        print(row.url)
        print(e)

100%|██████████| 12/12 [00:08<00:00,  1.39it/s]


In [ ]:
webqoof = pd.read_csv("webqoof_2022_2025.csv")

missing = pd.DataFrame(missing_articles)

webqoof = pd.concat([webqoof, missing], ignore_index=True)

# Remove duplicates just in case
webqoof = webqoof.drop_duplicates(subset="url")

webqoof.to_csv("webqoof_2022_2025.csv", index=False)

print(webqoof.shape)

(4737, 4)


In [ ]:
import pandas as pd

# Ensure the date column is datetime
webqoof["date"] = pd.to_datetime(webqoof["date"])

# Extract year
webqoof["year"] = webqoof["date"].dt.year

# Save one CSV per year
for year in [2022, 2023, 2024, 2025]:

    df_year = webqoof[webqoof["year"] == year]

    df_year.to_csv(
        f"webqoof_{year}.csv",
        index=False
    )

    print(f"{year}: {len(df_year)} articles saved.")

2022: 989 articles saved.
2023: 1093 articles saved.
2024: 1346 articles saved.
2025: 1309 articles saved.


In [ ]:
from google.colab import files

files.download("webqoof_2025.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>